# 경구약제 검출 — 정정판 파이프라인 (run_experiment_fixed)

기존 문제 진단:
- **검증셋 누수**: 같은 약 조합을 각도만 바꿔 찍은 사진이 train/val에 동시에 들어가
  val mAP가 0.9로 부풀려졌음(실제 일반화 성능 아님).
- **도메인/카테고리 불일치**: AIHub 임의 category_id로 학습 → 제출은 Kaggle K-코드 필요.

정정 내용:
1. **Kaggle 제공 train 데이터로 직접 학습**(테스트와 동일 분포, 올바른 category_id 자동 생성).
2. **조합(combo) 단위 그룹 분할**로 train/val 누수 0.
3. 전체 데이터 + 충분한 epoch.

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
PROJECT_ROOT

In [ ]:
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.subset import create_subset, combo_group_key
from src.data.coco_to_yolo import prepare_yolo_dataset
from src.train_yolo import run_yolo_experiment
from src.utils import load_config

CONFIGS_DIR = PROJECT_ROOT / "configs" / "joelchoi"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "joelchoi"
DATA_DIR = PROJECT_ROOT / "data" / "joelchoi"
YOLO_DIR = DATA_DIR / "yolo_kaggle_full"  # 정정판 데이터셋 출력 위치
print("PROJECT_ROOT:", PROJECT_ROOT)

## 1단계 — Kaggle train 데이터 → COCO
`category_id`는 K-코드 숫자(K-001900 → 1900)로 채점 기준과 동일. 조합2(테스트)는 train에 없음.

In [ ]:
coco = convert_kaggle_to_coco()
print("클래스 수:", len(coco["categories"]))

## 2단계 — 누수 없는 그룹 분할 + YOLO 데이터셋 생성
`create_subset`이 조합 접두사 단위로 분할(누수 0 자동 검증). `prepare_yolo_dataset`이
`data.yaml`과 `class_map.json`(yolo_idx → category_id)을 함께 생성.

In [ ]:
TIER = "full"  # 전체 데이터. 빠른 점검은 "medium"
train_coco, val_coco = create_subset(coco, tier=TIER, test_size=0.2, seed=42)


# 누수 재확인(조합 교집합 0이어야 함)
def get_combo_group_keys(content_data: dict) -> set:
    """이미지 목록에서 콤보 그룹 키 세트를 추출합니다."""
    return {combo_group_key(i["file_name"]) for i in content_data["images"]}


print(
    "train/val 공유 조합:",
    len(get_combo_group_keys(train_coco) & get_combo_group_keys(val_coco)),
)

yaml_path = prepare_yolo_dataset(train_coco, val_coco, YOLO_DIR, symlink=True)
CLASS_MAP = YOLO_DIR / "class_map.json"
print("data.yaml:", yaml_path)
print("class_map:", CLASS_MAP)

## 3단계 — 학습 (YOLO11n)

In [ ]:
cfg = load_config(CONFIGS_DIR / "experiment" / "exp010_yolo11n_kaggle.yaml")
cfg["data"]["subset"] = TIER
metrics = run_yolo_experiment(cfg, yaml_path, EXPERIMENTS_DIR)
print(metrics)  # 이제 val mAP는 '현실적' 수치 — 제출 점수와 비슷해야 정상

## 4단계 — 테스트 추론 & 제출 파일 생성

In [ ]:
from src.inference import run_inference, save_submission
from pathlib import Path

CLASS_MAP = YOLO_DIR / "class_map.json"

In [ ]:
EXP_NAME = cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / "joelchoi" / f"{EXP_NAME}_submission.csv"


In [ ]:
preds = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=cfg["data"]["img_size"],
)
save_submission(preds, SUBMISSION)
print("제출 파일:", SUBMISSION)

## 5단계 — 제출 파일 점검

In [ ]:
import pandas as pd

df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(df["image_id"].unique())
print(
    f"행 {len(df)}, 예측된 이미지 {len(pred_ids)}/{len(test_ids)}, 고유 category {df['category_id'].nunique()}"
)
print("검출 없는 이미지:", len(test_ids - pred_ids))
df.head()